# 58 · What the coordinate adds inside segments, and whether it replicates between specimens

Segment pseudobulk keeps three numbers per gene and specimen. A coordinate can add:

| Category | Feature | Discrete comparator |
|---|---|---|
| (iv) | **order of switch-on positions** inside a segment | the switch fraction r = (S2 − S1)/(S3 − S1), which gives a coarse order |
| (i) | within-segment gradients | invisible to segment means |
| (ii) | interior extrema (a peak or trough inside a segment) | attenuated by width ÷ segment length |
| (iii) | species shifts of a switch relative to each species' own segment boundaries | r confounds location with width |

Every feature is measured **per specimen**, so it can be replicated between the two mice (and the
two human sections) without using the species contrast. Each specimen's segment units come from its
own reviewed S1→S2 and S2→S3 transitions.

This run does part (iv), the primary test of whether continuous analysis adds information beyond
segment means (framing F2 against F3). Parts (i)–(iii) follow in a later section.

The coordinate is a parameter (`--coordinate`, `--coordinate-column`, `--coordinate-label`,
`--construction-genes`, or the `PSEUDOSPACE_PT_COORDINATE*` variables). The count-split arm runs
with the SCF13 label only, because it needs notebook 35's refit on half the counts. The protocol was
frozen in workstream B's phase-1 plan (section 8) before this notebook ran.

Cohort: 2 male control mice and 2 cortex sections from 1 male human donor. Results are descriptive;
the two human sections are not donor replicates.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Curves | per specimen, unweighted cubic regression spline (notebook 12's basis, shared knots, 6 df) on the common grid (61 points) |
| Segment units | per specimen, logistic-regression transitions (notebook 37): S1 → [0, 1), S2 → [1, 2), S3 → [2, 3] between the ends of the common grid |
| Monotone-like | amplitude (max − min of the curve) ≥ 0.5 log units and \|Spearman(curve, grid)\| ≥ 0.9 |
| Onset | half-amplitude crossing, in the specimen's segment units; falling genes are flipped |
| Switch fraction r | (m2 − m1)/(m3 − m1) from the specimen's label-segment means; NaN when \|m3 − m1\| < 0.1 |
| Fold protection | genes outside the coordinate's construction genes (SCF13 and DPT13: notebook 13's 611 PCA genes; SCF13-ED: its 549) |
| **(iv) primary set** | mouse genes monotone in both mice with the same direction, onset in [1, 2) in both, r defined in both, fold-protected |
| **(iv) primary metric** | partial Spearman of onset(Ctrl1A2) with onset(Ctrl1A4) given r(Ctrl1A2) and r(Ctrl1A4); 2,000 gene bootstraps (seed 58) for a 95% interval |
| Count-split arm | notebook 35's coordinate refit on half A of the counts (ε = 0.5, seed 35); curves, onsets and r from half B; the same primary set without fold protection |
| **(iv) decision** | claim if the SCF13 partial ρ ≥ 0.3 with its interval excluding 0 **and** the count-split partial ρ ≥ 0.2 · abandon the onset claim if both are < 0.1 · inconclusive otherwise |
| Human | COR1 against MED1 within S2 and within S3, descriptive only |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--coordinate', type=Path, help='CSV with structure or structure_id and a position column')
parser.add_argument('--coordinate-column')
parser.add_argument('--coordinate-label')
parser.add_argument('--construction-genes', type=Path, help='one gene per line; genes that built the coordinate')
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = Path(args.coordinate or os.environ.get('PSEUDOSPACE_PT_COORDINATE')
                       or results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv').expanduser()
COORDINATE_COLUMN = args.coordinate_column or os.environ.get('PSEUDOSPACE_PT_COORDINATE_COLUMN') or 'position'
COORDINATE_LABEL = args.coordinate_label or os.environ.get('PSEUDOSPACE_PT_COORDINATE_LABEL') or 'scf13'
construction_source = args.construction_genes or os.environ.get('PSEUDOSPACE_CONSTRUCTION_GENES')
upstream13 = results_root / 'minimal_pt_scfates'
nb35_cache = results_root / 'pt_reconstruction_v2' / 'nb35' / 'stage_cache'
nb36_cache = results_root / 'pt_reconstruction_v2' / 'nb36' / 'stage_cache'
output = results_root / 'pt_pathway_continuous' / '58_descriptors' / COORDINATE_LABEL
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
for path in (coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '58.descriptors.4'  # Bump when a cached calculation changes.
BASIS_DF = 6
MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE = .5, .9, .1
MIN_PROMINENCE, INNER = .2, .2            # (ii) interior extrema
SLOPE_CLAIM, SLOPE_SPLIT = .3, .2           # (i) between-specimen agreement of within-segment slopes
SHIFT_GENES, SHIFT_PROGRAMS, PROGRAM_MIN_GENES = 10, 3, 3   # (iii)
N_NULL_REPS, NULL_SEEDS = 10, {'A': 5901, 'B': 6001}       # added control: semi-synthetic nulls
N_BOOT, BOOT_SEED = 2000, 58
CLAIM_PRIMARY, CLAIM_SPLIT, ABANDON = .3, .2, .1
SPLIT_EPSILON, SPLIT_SEED = .5, 35  # notebook 35
MICE, HUMANS = ('Ctrl1A2', 'Ctrl1A4'), ('HUK1_COR1', 'HUK1_MED1')
# Construction genes for the coordinates built in notebooks 13/36 (payload key 'construction').
CONSTRUCTION_ARM = {'scf13': 'arm_scf13_full', 'dpt13': 'arm_scf13_full', 'scf13_ed': 'arm_scf13_ed'}
print('Coordinate:', coordinate_file, f'[{COORDINATE_COLUMN}] as', COORDINATE_LABEL)
print('Results folder:', output)

## 2 · Structures, expression and construction genes

In [ ]:
import glob

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

from pseudospace import continuous_descriptors as descriptors_module
from pseudospace import coordinate_inputs as inputs_module
from pseudospace.continuous_descriptors import (blockwise_permutation, bootstrap_partial_spearman,
                                                coordinate_segments, interior_extrema, logistic_transitions,
                                                monotone_onsets, partial_spearman, segment_means, segment_units,
                                                specimen_curves, switch_fraction, within_segment_slopes)
from pseudospace.coordinate_inputs import common_support_mask, pt_inputs, read_coordinate
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots

coordinate = read_coordinate(coordinate_file, COORDINATE_COLUMN)
inputs = pt_inputs(results_root, data_root, coordinate)
position, specimen, segment = inputs['position'], inputs['specimen'], inputs['segment']
genes = pd.Index(inputs['genes'])
print(f'{len(position):,} structures · {len(genes):,} genes')

if construction_source:
    construction = set(Path(construction_source).read_text().split())
elif COORDINATE_LABEL in CONSTRUCTION_ARM:
    payload = [p for p in glob.glob(str(nb36_cache / f'{CONSTRUCTION_ARM[COORDINATE_LABEL]}__*.npz'))]
    if len(payload) != 1:
        raise FileNotFoundError(f'Notebook 36 payload {CONSTRUCTION_ARM[COORDINATE_LABEL]} not found.')
    construction = set(np.load(payload[0], allow_pickle=True)['construction'].astype(str))
else:
    construction = set()
    print('No construction-gene list: fold protection is not applied for this coordinate.')
fold_protected = ~genes.isin(sorted(construction))
print(f'{len(construction)} construction genes; {int(fold_protected.sum()):,} fold-protected genes in the universe')

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(descriptors_module.__file__), Path(inputs_module.__file__)])

## 3 · Per-specimen curves, segment units and segment means

In [ ]:


def descriptor_payload(Y, position, specimen, segment, grid, coord_segment=None):
    """Flat dict of arrays per specimen: curves on the grid, segment units, onsets, r and segment means."""
    knots = gam_internal_knots(position, basis_df=BASIS_DF)
    curves = specimen_curves(Y, position, specimen, grid, knots)
    transitions = logistic_transitions(position, segment, specimen).pivot(
        index='specimen', columns='transition', values='position')
    out = {'grid': np.asarray(grid, float), 'specimens': np.asarray(transitions.index, str),
           't12': transitions['S1→S2'].to_numpy(float), 't23': transitions['S2→S3'].to_numpy(float)}
    out['skipped'] = np.array([n for n in curves if not grid[0] < transitions.loc[n, 'S1→S2']
                               < transitions.loc[n, 'S2→S3'] < grid[-1]], dtype=str)
    slopes = within_segment_slopes(Y, position, specimen, segment)
    coord_slopes = within_segment_slopes(Y, position, specimen, coord_segment) if coord_segment is not None else None
    for name, curve in curves.items():
        if name in out['skipped']:  # transitions out of order on this coordinate: no segment units
            continue
        units = segment_units(grid, transitions.loc[name, 'S1→S2'], transitions.loc[name, 'S2→S3'],
                              grid[0], grid[-1])
        rows = specimen == name
        onsets = monotone_onsets(curve, units, min_amplitude=MIN_AMPLITUDE, min_abs_rho=MIN_ABS_RHO)
        means = segment_means(Y[rows], segment[rows])
        out[f'{name}|curves'], out[f'{name}|units'], out[f'{name}|means'] = curve, units, means
        out[f'{name}|r'] = switch_fraction(means, MIN_RANGE)
        out[f'{name}|slope_t'] = slopes[name].to_numpy(float)
        if coord_segment is not None:  # post hoc: the same summaries on coordinate-threshold segments
            out[f'{name}|r_coord'] = switch_fraction(segment_means(Y[rows], coord_segment[rows]), MIN_RANGE)
            out[f'{name}|slope_coord_t'] = coord_slopes[name].to_numpy(float)
        out[f'{name}|extremum_segment'], out[f'{name}|extremum_sign'] = interior_extrema(
            curve, units, min_amplitude=MIN_AMPLITUDE, min_prominence=MIN_PROMINENCE, inner=INNER)
        for column in onsets:
            out[f'{name}|{column}'] = onsets[column].to_numpy(float)
    return out


def unpack(payload):
    """Nested view of a descriptor payload: transitions table and one dict per specimen."""
    D = {'grid': payload['grid'],
         'transitions': pd.DataFrame({'S1→S2': payload['t12'], 'S2→S3': payload['t23']},
                                     index=pd.Index(payload['specimens'].astype(str), name='specimen'))}
    D['skipped'] = [str(n) for n in payload.get('skipped', [])]
    for name in D['transitions'].index:
        if name in D['skipped']:
            continue
        onsets = pd.DataFrame({c: payload[f'{name}|{c}'] for c in
                               ('amplitude', 'rho', 'monotone', 'direction', 'onset', 'width')})
        onsets['monotone'] = onsets.monotone.astype(bool)
        D[name] = {'curves': payload[f'{name}|curves'], 'units': payload[f'{name}|units'],
                   'means': payload[f'{name}|means'], 'r': payload[f'{name}|r'], 'onsets': onsets,
                   'slope_t': payload[f'{name}|slope_t'], 'extremum_segment': payload[f'{name}|extremum_segment'],
                   'extremum_sign': payload[f'{name}|extremum_sign'],
                   'r_coord': payload.get(f'{name}|r_coord'), 'slope_coord_t': payload.get(f'{name}|slope_coord_t')}
    if 'n_structures' in payload:
        D['n_structures'] = int(payload['n_structures'])
    return D


coord_segment, thresholds = coordinate_segments(position, segment, specimen)
D = unpack(cached_payload('descriptors', lambda: descriptor_payload(inputs['Y'], position, specimen, segment,
                                                                     inputs['grid'], coord_segment),
                          root=cache, params={'basis_df': BASIS_DF, 'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE]},
                          inputs={'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen,
                                  'segment': segment}, code=code))
transitions = D['transitions']
display(transitions.round(3))
if D['skipped']:
    raise ValueError(f"Transitions out of order on the primary coordinate for {D['skipped']}.")
monotone_counts = pd.DataFrame({name: {'monotone-like genes': int(D[name]['onsets'].monotone.sum()),
                                       'onset in S1': int((D[name]['onsets'].onset < 1).sum()),
                                       'onset in S2': int(D[name]['onsets'].onset.between(1, 2, inclusive='left').sum()),
                                       'onset in S3': int((D[name]['onsets'].onset >= 2).sum())}
                                for name in MICE + HUMANS}).T
display(monotone_counts)

## 4 · (iv) Does the order of switch-on positions inside a segment replicate beyond r?

The partial correlation asks whether the two specimens agree on where genes switch on **after**
removing what each specimen's three segment means already say (r). Genes are treated as
independent in the bootstrap, so the interval is optimistic.

In [ ]:


def onset_table(D, pair, genes):
    a, b = D[pair[0]], D[pair[1]]
    table = pd.DataFrame({'onset_a': a['onsets'].onset.to_numpy(), 'onset_b': b['onsets'].onset.to_numpy(),
                          'dir_a': a['onsets'].direction.to_numpy(), 'dir_b': b['onsets'].direction.to_numpy(),
                          'mono_a': a['onsets'].monotone.to_numpy(), 'mono_b': b['onsets'].monotone.to_numpy(),
                          'width_a': a['onsets'].width.to_numpy(), 'width_b': b['onsets'].width.to_numpy(),
                          'r_a': a['r'], 'r_b': b['r'],
                          'rc_a': a['r_coord'] if a['r_coord'] is not None else np.nan,
                          'rc_b': b['r_coord'] if b['r_coord'] is not None else np.nan}, index=genes)
    return table


def ordering_test(table, segment_index, keep=None, seed=BOOT_SEED, covariates=('r_a', 'r_b')):
    lo, hi = segment_index, segment_index + 1
    m = (table.mono_a & table.mono_b & table.dir_a.eq(table.dir_b)
         & table.onset_a.ge(lo) & table.onset_a.lt(hi) & table.onset_b.ge(lo) & table.onset_b.lt(hi)
         & table[list(covariates)].notna().all(axis=1))
    if keep is not None:
        m &= keep
    t = table[m]
    row = {'genes': int(m.sum())}
    if len(t) >= 20:
        est, low, high = bootstrap_partial_spearman(t.onset_a, t.onset_b, t[list(covariates)].to_numpy(),
                                                    n_boot=N_BOOT, seed=seed)
        row.update({'partial Spearman (onset | r)': est, '95% low': low, '95% high': high,
                    'Spearman onset a vs b': spearmanr(t.onset_a, t.onset_b).statistic,
                    'Spearman r a vs b': spearmanr(t.r_a, t.r_b).statistic,
                    'Spearman onset vs r (a)': spearmanr(t.onset_a, -t.r_a).statistic,
                    'onset IQR (a)': float(t.onset_a.quantile(.75) - t.onset_a.quantile(.25)),
                    'median |onset a − b|': float((t.onset_a - t.onset_b).abs().median())})
    return row, t


mouse_table = onset_table(D, MICE, genes)
human_table = onset_table(D, HUMANS, genes)
ordering_rows = []
primary_row, primary_genes = ordering_test(mouse_table, 1, keep=fold_protected)
ordering_rows.append({'arm': COORDINATE_LABEL, 'pair': 'mouse', 'segment': 'S2', 'genes used': 'fold-protected', **primary_row})
row, _ = ordering_test(mouse_table, 1)
ordering_rows.append({'arm': COORDINATE_LABEL, 'pair': 'mouse', 'segment': 'S2', 'genes used': 'all', **row})
for seg_index, seg_name in ((0, 'S1'), (2, 'S3')):
    row, _ = ordering_test(mouse_table, seg_index, keep=fold_protected)
    ordering_rows.append({'arm': COORDINATE_LABEL, 'pair': 'mouse', 'segment': seg_name, 'genes used': 'fold-protected', **row})
for seg_index, seg_name in ((1, 'S2'), (2, 'S3')):
    row, _ = ordering_test(human_table, seg_index, keep=fold_protected)
    ordering_rows.append({'arm': COORDINATE_LABEL, 'pair': 'human (one donor)', 'segment': seg_name,
                          'genes used': 'fold-protected', **row})
primary_genes.to_csv(tables / 'iv_primary_mouse_s2_onsets.csv')

## 5 · (iv) Count-split arm (SCF13 only)

Notebook 35 split every count into A ~ Binomial(x, 0.5) and B = x − A and refit the coordinate on
A. The nephron counts are rebuilt here, checked against notebook 35's recorded digest, and split
with the same seed. Onsets and r then come from half B, so no structure's position was built from
the counts that describe it.

In [ ]:
split_row, DS = None, None
split_payload = sorted(glob.glob(str(nb35_cache / 'refit_split_A__*.npz')))
if COORDINATE_LABEL == 'scf13' and len(split_payload) == 1:
    import anndata as ad
    from scipy import sparse

    from pseudospace.coordinate_evaluation import log_normalize, poisson_count_split
    from pseudospace.pathway_inputs import rebuild_pt_expression

    def split_descriptors():
        saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
        saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
        source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in MICE + HUMANS}
        rebuilt = rebuild_pt_expression(saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype(
            {'sample': str, 'comparison_species': str}), source_paths,
            pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
        counts = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
        recorded = json.loads(Path(split_payload[0] + '.key.json').read_text())['inputs']
        if f"'base_counts': '{digest(counts)}'" not in recorded:  # Guard: same counts as notebook 35.
            raise ValueError('Rebuilt nephron counts differ from notebook 35; the split would not match.')
        _, half_b = poisson_count_split(counts, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))
        pt_rows = saved_neph.index.get_indexer(saved_pt.index)
        if (pt_rows < 0).any():
            raise ValueError('PT structures missing from the nephron object.')
        measured = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
        block = half_b[pt_rows]
        library = np.asarray(block[:, measured].sum(axis=1)).ravel().astype(float)
        cols = pd.Index(rebuilt.var_names).get_indexer(genes)
        keep_rows = library > 0
        expr = log_normalize(block[:, cols], np.where(keep_rows, library, 1.))
        z = np.load(split_payload[0], allow_pickle=True)['scfates'].astype(float)
        spec = saved_pt['sample'].astype(str).to_numpy(dtype=str)
        seg = saved_pt.segment_class.astype(str).to_numpy(dtype=str)
        support = common_support_mask(z, spec) & keep_rows
        split_coord_segment, _ = coordinate_segments(z[support], seg[support], spec[support])
        names = MICE + HUMANS
        lo = max(z[support & (spec == n)].min() for n in names)
        hi = min(z[support & (spec == n)].max() for n in names)
        result = descriptor_payload(sparse.csr_matrix(expr)[support], z[support], spec[support], seg[support],
                                    np.linspace(lo, hi, 61), split_coord_segment)
        result['n_structures'] = int(support.sum())
        return result

    DS = unpack(cached_payload('descriptors_count_split', split_descriptors,
                            root=cache, params={'epsilon': SPLIT_EPSILON, 'seed': SPLIT_SEED, 'basis_df': BASIS_DF,
                                                'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE]},
                            inputs={'payload': Path(split_payload[0]).name, 'genes': list(genes)}, code=code))
    print('Count-split coordinate transitions:'), display(DS['transitions'].round(3))
    print('Specimens without ordered transitions on the count-split coordinate:', DS['skipped'] or 'none')
    split_table = onset_table(DS, MICE, genes)
    split_row, split_genes = ordering_test(split_table, 1)
    ordering_rows.append({'arm': 'count split (coordinate from A, data from B)', 'pair': 'mouse', 'segment': 'S2',
                          'genes used': 'all', **split_row})
    split_construction = set(np.load(split_payload[0], allow_pickle=True)['pca_genes'].astype(str))
    row, _ = ordering_test(split_table, 1, keep=~genes.isin(sorted(split_construction)))
    ordering_rows.append({'arm': 'count split (coordinate from A, data from B)', 'pair': 'mouse', 'segment': 'S2',
                          'genes used': 'outside split-A construction genes', **row})
    split_genes.to_csv(tables / 'iv_count_split_mouse_s2_onsets.csv')
    print(f"Count-split arm: {DS['n_structures']:,} structures in common support")
else:
    print('Count-split arm not run for this coordinate.')
ORDERING = pd.DataFrame(ordering_rows)
ORDERING.to_csv(tables / 'iv_onset_ordering.csv', index=False)
display(ORDERING.round(3))

## 6 · (iv) Decision

In [ ]:
primary = primary_row.get('partial Spearman (onset | r)', np.nan)
primary_low = primary_row.get('95% low', np.nan)
split_value = split_row.get('partial Spearman (onset | r)', np.nan) if split_row else np.nan
if COORDINATE_LABEL != 'scf13':
    decision_iv = 'sensitivity coordinate: reported against the SCF13 decision'
elif primary >= CLAIM_PRIMARY and primary_low > 0 and split_value >= CLAIM_SPLIT:
    decision_iv = 'claim: onset order inside mouse S2 replicates between mice beyond r'
elif primary < ABANDON and split_value < ABANDON:
    decision_iv = 'abandon: onset order adds nothing reproducible beyond r'
else:
    decision_iv = 'inconclusive'
decision = pd.Series({'coordinate': COORDINATE_LABEL, 'primary partial ρ (fold-protected, mouse S2)': primary,
                      'primary 95% interval': f"{primary_low:.3f} to {primary_row.get('95% high', np.nan):.3f}",
                      'primary genes': primary_row['genes'], 'count-split partial ρ': split_value,
                      'count-split genes': split_row['genes'] if split_row else 0,
                      'decision (iv)': decision_iv}, name='value')
display(decision.to_frame())
decision.to_csv(tables / 'iv_decision.csv')

## 7 · Added control: what pure segment steps produce (semi-synthetic nulls)

**Not in the phase-1 protocol; added before parts (i)–(iii) were computed.** Notebook 57 showed that
label–coordinate discordance alone produces about half of the "beyond steps" pathway calls. The same
could hold for the descriptors here. A spline fitted to a step produces a ramp, an overshoot near the
boundary and a gene-specific onset that depends on the step sizes, and label noise turns steps into
within-label slopes. The nulls permute expression rows within specimen × coordinate-threshold segment
(arm A: truth = steps on the coordinate) or within specimen × label (arm B: truth = label steps), then
recompute every descriptor. Ten replicates per arm. An observed statistic counts as beyond steps only
if it exceeds every null replicate; the pre-specified rules are reported unchanged beside it.

**Post hoc variants, added after the first null comparison showed that coordinate steps plus label
noise reproduce the label-based "beyond r" signal.** (iv-b) repeats the onset test given r computed
from coordinate-threshold segments, and (i-b) computes slopes within coordinate-threshold segments.
They ask whether the coordinate carries information beyond its own segmentation. Arm A is their
exact null; arm B is the null for coordinate noise around label steps.

In [ ]:
null_blocks = {'A': np.array([f'{a}|{b}' for a, b in zip(specimen, coord_segment)]),
               'B': np.array([f'{a}|{b}' for a, b in zip(specimen, segment)])}


def slope_agreement(D, pair, keep, key='slope_t'):
    a, b = D[pair[0]][key], D[pair[1]][key]
    out = {}
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        m = keep & np.isfinite(a[:, k]) & np.isfinite(b[:, k])
        out[seg] = spearmanr(a[m, k], b[m, k]).statistic if m.sum() > 50 else np.nan
    return out


def replicated_extrema(D, pair):
    a, b = D[pair[0]], D[pair[1]]
    same = (a['extremum_segment'] >= 0) & (a['extremum_segment'] == b['extremum_segment']) & (a['extremum_sign'] == b['extremum_sign'])
    return same, {seg: int((same & (a['extremum_segment'] == k)).sum()) for k, seg in enumerate(('S1', 'S2', 'S3'))}


def descriptor_metrics(D):
    row = {}
    t = onset_table(D, MICE, genes)
    r_iv, _ = ordering_test(t, 1, keep=fold_protected)
    row['(iv) mouse S2 partial ρ'] = r_iv.get('partial Spearman (onset | r)', np.nan)
    row['(iv) mouse S2 genes'] = r_iv['genes']
    r_ivc, _ = ordering_test(t, 1, keep=fold_protected, covariates=('rc_a', 'rc_b'))
    row['(iv-b) mouse S2 partial ρ given coordinate-segment r'] = r_ivc.get('partial Spearman (onset | r)', np.nan)
    for species_name, pair in (('mouse', MICE), ('human', HUMANS)):
        for seg, value in slope_agreement(D, pair, fold_protected).items():
            row[f'(i) {species_name} {seg} slope ρ'] = value
        for seg, value in slope_agreement(D, pair, fold_protected, 'slope_coord_t').items():
            row[f'(i-b) {species_name} {seg} slope ρ within coordinate segments'] = value
        _, counts = replicated_extrema(D, pair)
        for seg, value in counts.items():
            row[f'(ii) {species_name} {seg} replicated extrema'] = value
    return row


def null_replicate(arm, rep):
    index = blockwise_permutation(null_blocks[arm], np.random.default_rng(NULL_SEEDS[arm] + rep))
    Dn = unpack(descriptor_payload(inputs['Y'][index], position, specimen, segment, inputs['grid'], coord_segment))
    return pd.DataFrame([{'arm': arm, 'replicate': rep, **descriptor_metrics(Dn)}])


NULLS = pd.concat([cached_frame(f'null_{arm}_{rep}', lambda arm=arm, rep=rep: null_replicate(arm, rep), root=cache,
                                params={'arm': arm, 'rep': rep, 'seed': NULL_SEEDS[arm]},
                                inputs={'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen,
                                        'segment': segment, 'coord_segment': coord_segment}, code=code)
                   for arm in ('A', 'B') for rep in range(1, N_NULL_REPS + 1)], ignore_index=True)
OBSERVED = pd.Series(descriptor_metrics(D), name='observed')
comparison = pd.DataFrame({'observed': OBSERVED,
                           'arm A max': NULLS[NULLS.arm.eq('A')].drop(columns=['arm', 'replicate']).max(),
                           'arm A median': NULLS[NULLS.arm.eq('A')].drop(columns=['arm', 'replicate']).median(),
                           'arm B max': NULLS[NULLS.arm.eq('B')].drop(columns=['arm', 'replicate']).max(),
                           'arm B median': NULLS[NULLS.arm.eq('B')].drop(columns=['arm', 'replicate']).median()})
comparison['exceeds every null replicate'] = comparison.observed > comparison[['arm A max', 'arm B max']].max(axis=1)
NULLS.to_csv(tables / 'null_replicates.csv', index=False)
comparison.to_csv(tables / 'observed_against_step_nulls.csv')
display(comparison.round(3))

## 8 · (i) Within-segment gradients

Per specimen and label segment, each gene's slope t along the coordinate; agreement between the two
specimens of a species over fold-protected genes. Pre-specified: claimed for a species × segment cell
if ρ ≥ 0.3 on SCF13 and ≥ 0.2 under count split. The count-split coordinate does not order the
segments well (section 5), so the count-split value is reported where it exists.

In [ ]:
gradient_rows = []
for species_name, pair in (('mouse', MICE), ('human', HUMANS)):
    observed_rho = slope_agreement(D, pair, fold_protected)
    split_rho = slope_agreement(DS, pair, np.ones(len(genes), bool)) if (
        DS is not None and all(n in DS for n in pair)) else {}
    for seg, value in observed_rho.items():
        key = f'(i) {species_name} {seg} slope ρ'
        claimed = (COORDINATE_LABEL == 'scf13' and value >= SLOPE_CLAIM and split_rho.get(seg, np.nan) >= SLOPE_SPLIT)
        gradient_rows.append({'species': species_name, 'segment': seg, 'slope ρ (fold-protected)': value,
                              'count-split slope ρ': split_rho.get(seg, np.nan),
                              'arm A max': comparison.loc[key, 'arm A max'], 'arm B max': comparison.loc[key, 'arm B max'],
                              'exceeds every null replicate': bool(comparison.loc[key, 'exceeds every null replicate']),
                              'pre-specified claim': claimed})
GRADIENTS = pd.DataFrame(gradient_rows)
slope_columns = {}
for n in MICE + HUMANS:
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        slope_columns[f'{n} {seg} label'] = D[n]['slope_t'][:, k]
        slope_columns[f'{n} {seg} coordinate'] = D[n]['slope_coord_t'][:, k]
pd.DataFrame(slope_columns, index=genes).assign(fold_protected=fold_protected).to_csv(tables / 'i_slopes_per_gene.csv')
GRADIENTS.to_csv(tables / 'i_within_segment_gradients.csv', index=False)
display(GRADIENTS.round(3))

## 9 · (ii) Interior extrema

A gene has a replicated interior extremum when both specimens of a species put its peak (or trough)
in the inner 60% of the same segment, with prominence ≥ 0.2 log units above both segment ends.
Agreement across coordinates and with the equal-depth refit is assessed in notebook 60.

In [ ]:
extrema_rows = []
for species_name, pair in (('mouse', MICE), ('human', HUMANS)):
    same, _ = replicated_extrema(D, pair)
    seg_index = D[pair[0]]['extremum_segment']
    for i in np.flatnonzero(same):
        extrema_rows.append({'gene': genes[i], 'species': species_name, 'segment': ('S1', 'S2', 'S3')[seg_index[i]],
                             'kind': 'peak' if D[pair[0]]['extremum_sign'][i] > 0 else 'trough',
                             'fold-protected': bool(fold_protected[i])})
EXTREMA = pd.DataFrame(extrema_rows, columns=['gene', 'species', 'segment', 'kind', 'fold-protected'])
EXTREMA.to_csv(tables / 'ii_replicated_interior_extrema.csv', index=False)
display(EXTREMA.groupby(['species', 'segment', 'kind']).size().rename('replicated genes').to_frame())
display(comparison.filter(like='(ii)', axis=0).round(1))

## 10 · (iii) Species shifts of a switch relative to each species' own segment boundaries

For genes monotone with the same direction in all four specimens, shift = mean human onset − mean
mouse onset, each in its own segment units. A gene passes when all four cross-species specimen pairs
agree in sign and \|shift\| exceeds the 95th percentile of the registration null: the change in a
human onset's segment units when human positions in its segment move by the change in the
species registration gap across notebook 35's refits (SCF13 rows). Pre-specified claim: ≥ 10 genes
or ≥ 3 notebook-37 programs (≥ 3 driver genes shifted with one sign).

In [ ]:
registration_file = results_root / 'pt_reconstruction_v2' / 'nb35' / 'species_registration_across_refits.csv'
four = MICE + HUMANS
onset = pd.DataFrame({n: D[n]['onsets'].onset.to_numpy() for n in four}, index=genes)
direction = pd.DataFrame({n: D[n]['onsets'].direction.to_numpy() for n in four}, index=genes)
mono = pd.DataFrame({n: D[n]['onsets'].monotone.to_numpy() for n in four}, index=genes)
eligible = mono.all(axis=1) & direction.nunique(axis=1).eq(1)
cross_signs = pd.concat([np.sign(onset[h] - onset[m]) for h in HUMANS for m in MICE], axis=1)
consistent = cross_signs.nunique(axis=1).eq(1) & cross_signs.iloc[:, 0].ne(0)
shift = onset[list(HUMANS)].mean(axis=1) - onset[list(MICE)].mean(axis=1)
human_segment = np.floor(onset[list(HUMANS)].mean(axis=1).clip(0, 2.999)).astype('Int64')
threshold_by_segment = {}
if registration_file.is_file():
    registration = pd.read_csv(registration_file)
    registration = registration[registration.coordinate.eq('SCF13')]
    saved_gap = registration[registration.refit.eq('saved')].iloc[0]
    refits = registration[~registration.refit.eq('saved')]
    human_lengths = []
    for n in HUMANS:
        t12, t23 = D['transitions'].loc[n, 'S1→S2'], D['transitions'].loc[n, 'S2→S3']
        human_lengths.append([t12 - D['grid'][0], t23 - t12, D['grid'][-1] - t23])
    lengths = np.mean(human_lengths, axis=0)
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        moves = (refits[f'{seg} human−mouse'] - saved_gap[f'{seg} human−mouse']).abs() / lengths[k]
        threshold_by_segment[k] = float(moves.quantile(.95))
threshold = human_segment.map(threshold_by_segment).astype(float)
passes = eligible & consistent & (shift.abs() > threshold)
SHIFTS = pd.DataFrame({'shift (segment units)': shift, 'human onset segment': human_segment,
                       'registration threshold': threshold, 'four-pair sign agreement': consistent,
                       'passes': passes})[eligible]
SHIFTS.to_csv(tables / 'iii_species_boundary_shifts.csv')
programs = pd.read_csv(results_root / 'pt_pathway_final' / 'scfates' / 'tables' / 'table_S4_programs.csv')
program_rows = []
for row in programs.itertuples():
    drivers = [g.strip() for g in str(row.drivers).split(',')]
    shifted = [g for g in drivers if g in SHIFTS.index and SHIFTS.loc[g, 'passes']]
    signs = {np.sign(SHIFTS.loc[g, 'shift (segment units)']) for g in shifted}
    program_rows.append({'program': row.name, 'drivers': len(drivers), 'shifted drivers': len(shifted),
                         'one sign': len(signs) == 1, 'passes': len(shifted) >= PROGRAM_MIN_GENES and len(signs) == 1,
                         'genes': ', '.join(shifted)})
PROGRAM_SHIFTS = pd.DataFrame(program_rows)
PROGRAM_SHIFTS.to_csv(tables / 'iii_program_shifts.csv', index=False)
shift_claim = int(passes.sum()) >= SHIFT_GENES or int(PROGRAM_SHIFTS.passes.sum()) >= SHIFT_PROGRAMS
print(f'Registration thresholds (segment units) by human onset segment: '
      f'{ {k: round(v, 2) for k, v in threshold_by_segment.items()} }')
print(f'{int(eligible.sum())} genes monotone in all four specimens; {int((eligible & consistent).sum())} with four-pair '
      f'sign agreement; {int(passes.sum())} beyond the registration null; programs passing: {int(PROGRAM_SHIFTS.passes.sum())}')
display(SHIFTS[SHIFTS.passes].sort_values('shift (segment units)').round(2))

## 11 · Summary for this coordinate

In [ ]:
summary = pd.Series({
    'coordinate': COORDINATE_LABEL,
    '(iv) mouse S2 partial ρ (fold-protected)': primary,
    '(iv) interval': f"{primary_low:.2f} to {primary_row.get('95% high', np.nan):.2f}",
    '(iv) exceeds every step-null replicate': bool(comparison.loc['(iv) mouse S2 partial ρ', 'exceeds every null replicate']),
    '(iv) pre-specified decision': decision_iv,
    '(iv-b, post hoc) partial ρ given coordinate-segment r': OBSERVED['(iv-b) mouse S2 partial ρ given coordinate-segment r'],
    '(iv-b) step-null max': comparison.loc['(iv-b) mouse S2 partial ρ given coordinate-segment r', ['arm A max', 'arm B max']].max(),
    '(i-b, post hoc) cells beyond every step-null replicate': ', '.join(
        k.split()[1] + ' ' + k.split()[2] for k in comparison.index
        if k.startswith('(i-b)') and comparison.loc[k, 'exceeds every null replicate']) or 'none',
    '(i) cells with slope ρ ≥ 0.3': ', '.join(f'{r.species} {r.segment}' for r in GRADIENTS.itertuples()
                                               if r[3] >= SLOPE_CLAIM) or 'none',
    '(i) cells beyond every step-null replicate': ', '.join(f'{r.species} {r.segment}' for r in GRADIENTS.itertuples()
                                                            if r[7]) or 'none',
    '(ii) replicated interior extrema (mouse, human)': f"{int((EXTREMA.species == 'mouse').sum())}, {int((EXTREMA.species == 'human').sum())}",
    '(iii) genes passing / programs passing': f'{int(passes.sum())} / {int(PROGRAM_SHIFTS.passes.sum())}',
    '(iii) pre-specified claim': shift_claim}, name='value')
summary.to_csv(tables / 'summary.csv')
display(summary.to_frame())
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'coordinate_file': coordinate_file.name,
          'coordinate_column': COORDINATE_COLUMN, 'coordinate_label': COORDINATE_LABEL,
          'n_structures': int(len(position)), 'n_genes': int(len(genes)), 'n_construction_genes': len(construction),
          'summary': {k: (v if isinstance(v, (str, bool)) else float(v)) for k, v in summary.items()}}
(output / 'run_record.json').write_text(json.dumps(record, indent=2))